[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage1_c_subword_reference.ipynb)

# Stage 1c: Zero-Layer Transformer (Subword/BPE-Level) 🧩
## target: Visualize the 3D Dialect Fracture in LLM Embeddings

We've covered characters and words. Now let's explore **Subword Tokenization (BPE)**, which is the actual tokenization scheme used by real-world Large Language Models (like GPT-4, LLaMA, or mGPT).

Because BPE tokenization chunks words into subword pieces based on frequency, it can introduce a **"dialect tax"**. Masri phrases, being less common in standard pre-training sets, are often fractured into many more fragments than their MSA equivalents.

### 💡 What we are showing:
We will load `mGPT`'s tokenizer and its pre-trained embedding weights. We will perform dimensionality reduction (3D PCA) on the embeddings of words and subwords, and visualize how the embedding space naturally clusters or separates MSA and Masri concepts!


In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformers plotly pandas scikit-learn datasets


In [ ]:
# 📦 Load pre-trained mGPT Tokenizer & Embeddings
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "ai-forever/mGPT"
print(f"Loading {model_name} tokenizer & model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Extract embedding weights
W_E = model.wte.weight.data.numpy()
print(f"Embedding matrix shape: {W_E.shape}") # [vocab_size, hidden_dim]


## 🕵️‍♂️ Dim-Reduction: 3D PCA
To visualize how Egyptian/Masri and MSA are represented, we will select common words in both dialects (e.g., MSA: 'ماذا', 'هنا', 'الآن', 'المرأة', 'السيارة', 'الكلب' vs Masri: 'إيه', 'هنا', 'دلوقتي', 'الست', 'العربية', 'الكلب').
We'll tokenize these words, extract their embedding vectors from $W_E$, apply PCA to reduce the dimensionality to 3 dimensions, and plot them in 3D space!


In [ ]:
# 🎨 3D PCA Clustering of Dialect Embeddings
from sklearn.decomposition import PCA
import pandas as pd
import plotly.express as px

# Define word pairs (MSA vs Masri equivalents)
words = [
    # MSA
    ("ماذا", "MSA"), ("لماذا", "MSA"), ("الآن", "MSA"), ("المرأة", "MSA"), ("السيارة", "MSA"), ("حذاء", "MSA"), ("كيف", "MSA"),
    # Masri
    ("إيه", "Masri"), ("ليه", "Masri"), ("دلوقتي", "Masri"), ("الست", "Masri"), ("العربية", "Masri"), ("جزمة", "Masri"), ("إزاي", "Masri"),
    # Shared
    ("هنا", "Shared"), ("الكلب", "Shared"), ("كتاب", "Shared"), ("شمس", "Shared"), ("قمر", "Shared")
]

# Extract embedding vectors
vectors = []
labels = []
categories = []

for word, cat in words:
    # Tokenize word
    tokens = tokenizer.encode(word, add_special_tokens=False)
    # Average the embeddings of the subwords if fractured
    word_vector = W_E[tokens].mean(axis=0)
    
    vectors.append(word_vector)
    labels.append(word)
    categories.append(cat)

# Perform 3D PCA
pca = PCA(n_components=3)
reduced_vectors = pca.fit_transform(vectors)

# Create DataFrame
df = pd.DataFrame(reduced_vectors, columns=['PC1', 'PC2', 'PC3'])
df['Word'] = labels
df['Dialect'] = categories

# Plotly 3D Scatter
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    text='Word', color='Dialect',
    title="3D PCA of MSA vs Masri Word Embeddings (mGPT)",
    color_discrete_map={"MSA": "red", "Masri": "blue", "Shared": "green"}
)
fig.update_traces(textposition='top center')
fig.update_layout(width=800, height=800)
fig.show()
